# Comparing Growth Rates by Fitting Exponential Curves
This notebook is intended to be used for comparing growth rates from incucyte data. For example, the growth rate of a fusion clone versus a control clone.

- Use environment.yml

## Import Necessary Modules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

## Functions

In [ ]:
""" Function to remove first (header) line in a txt file and save the output as a new txt file.
The new first line should be the column headers.
Function inputs, input file path and output file path respectively. """
def remove_first_txt_line(in_file_path,out_file_path):
    input_file_path = in_file_path
    output_file_path = out_file_path

    with open(input_file_path, 'r') as input_file:
        lines = input_file.readlines()[1:]

    with open(output_file_path, 'w') as output_file:
        for line in lines:
            output_file.write(line)

In [ ]:
def import_incucyte_sample(og_file, keep_all_columns=False):
    """
    Full import pipeline for one incucyte sample file:
    - strips the first (title) line and writes a '_firstlineremoved.txt' copy
    - loads the result into a DataFrame
    - trims to well-columns-only (drops 'Elapsed'/'Date Time') unless keep_all_columns=True

    Parameters:
    og_file (str): path to the original incucyte txt export.
    keep_all_columns (bool): set True for exactly one sample (the first one going
        into the concat) so 'Elapsed'/'Date Time' survive into the merged dataframe.

    Returns:
    pd.DataFrame
    """
    out_file = og_file.replace('.txt', '_firstlineremoved.txt')
    remove_first_txt_line(og_file, out_file)
    df = pd.read_csv(out_file, delimiter='\t')
    if not keep_all_columns:
        df = df.iloc[:, 2:]
    return df

In [ ]:
def plot_variable_vs_time(dataframe, sample_ID_df, xlabel=None, ylabel=None, grid_lines=True, title=None, plot_separately=False, save_svg=False, fig_size=(10, 6), font_sz=18):
    
    # --- Publication style settings ---
    # mpl.rcParams.update({
    #     'font.family': 'Arial',
    #     'font.weight': 'bold',
    #     'axes.labelweight': 'bold',
    #     'axes.titleweight': 'bold',
    #     'font.size': font_sz,
    #     'axes.titlesize': font_sz,
    #     'axes.labelsize': font_sz,
    #     'xtick.labelsize': font_sz - 2,
    #     'ytick.labelsize': font_sz - 2,
    #     'legend.fontsize': font_sz - 8,
    #     'axes.linewidth': 1.5,
    #     'xtick.major.width': 1.5,
    #     'ytick.major.width': 1.5,
    #     'xtick.major.size': 5,
    #     'ytick.major.size': 5,
    #     'lines.linewidth': 2,
    #     'errorbar.capsize': 4,
    # })

    dataframe = dataframe.apply(pd.to_numeric, errors='coerce')
    time_values = dataframe.loc[:, 'Elapsed']

    max_y = dataframe.drop(['Elapsed', 'Date Time'], axis=1, errors='ignore').max().max()
    y_padding = (max_y) * 0.1

    def style_axes(ax):
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1.5)
        ax.spines['bottom'].set_linewidth(1.5)
        ax.yaxis.set_ticks_position('left')
        ax.xaxis.set_ticks_position('bottom')
        if grid_lines:
            ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.5, color='gray')
        ax.set_axisbelow(True)

    def save_and_show(fig, plot_title, save_svg):
        if save_svg:
            filename = f"{plot_title}.svg" if plot_title else "plot.svg"
            fig.savefig(filename, format='svg', bbox_inches='tight', dpi=300)
        plt.show()

    yerr_df = pd.DataFrame()

    # --- Combined plot ---
    fig, ax = plt.subplots(figsize=fig_size)
    ax.set_ylim(0, max_y + y_padding)
    style_axes(ax)

    if any('SE' in column for column in dataframe.columns):
        for column in dataframe.columns:
            for sample in sample_ID_df.columns:
                if 'SE' in column and sample_ID_df.at['key', sample] in column:
                    yerr_df[sample] = dataframe[column].values

        for column in dataframe.columns:
            for sample in sample_ID_df.columns:
                if sample_ID_df.at['key', sample] in column and 'SE' not in column:
                    ax.errorbar(time_values, dataframe[column], yerr=yerr_df[sample],
                                color=sample_ID_df.at['color', sample], label=column,
                                linewidth=2, capsize=4, capthick=1.5, elinewidth=1.5)
    else:
        for column in dataframe.columns:
            for sample in sample_ID_df.columns:
                if sample_ID_df.at['key', sample] in column:
                    ax.plot(time_values, dataframe[column], label=column,
                            color=sample_ID_df.at['color', sample], linewidth=2)

    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)

    if xlabel:
        ax.set_xlabel(xlabel, labelpad=10)
    if ylabel:
        ax.set_ylabel(ylabel, labelpad=10)
    if title:
        ax.set_title(title, pad=12, fontweight='bold')

    fig.tight_layout()
    save_and_show(fig, title, save_svg)

    # --- Separate plots ---
    if plot_separately:
        if any('SE' in column for column in dataframe.columns):
            for column in dataframe.columns:
                for sample in sample_ID_df.columns:
                    if 'SE' in column and sample_ID_df.at['key', sample] in column:
                        yerr_df[sample] = dataframe[column].values

            for column in dataframe.columns:
                for sample in sample_ID_df.columns:
                    if sample_ID_df.at['key', sample] in column and 'SE' not in column:
                        fig, ax = plt.subplots(figsize=fig_size)
                        ax.errorbar(time_values, dataframe[column], yerr=yerr_df[sample],
                                    color=sample_ID_df.at['color', sample], label=column,
                                    linewidth=2, capsize=4, capthick=1.5, elinewidth=1.5)
                        ax.set_ylim(0, max_y + y_padding)
                        style_axes(ax)
                        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
                        if xlabel: ax.set_xlabel(xlabel, labelpad=10)
                        if ylabel: ax.set_ylabel(ylabel, labelpad=10)
                        plot_title = title + f' - {column}' if title else f'Plot for {column}'
                        ax.set_title(plot_title, pad=12, fontweight='bold')
                        fig.tight_layout()
                        save_and_show(fig, plot_title, save_svg)
        else:
            for column in dataframe.columns:
                for sample in sample_ID_df.columns:
                    if sample_ID_df.at['key', sample] in column:
                        fig, ax = plt.subplots(figsize=fig_size)
                        ax.plot(time_values, dataframe[column], label=column,
                                color=sample_ID_df.at['color', sample], linewidth=2)
                        ax.set_ylim(0, max_y + y_padding)
                        style_axes(ax)
                        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
                        if xlabel: ax.set_xlabel(xlabel, labelpad=10)
                        if ylabel: ax.set_ylabel(ylabel, labelpad=10)
                        plot_title = title + f' - {column}' if title else f'Plot for {column}'
                        ax.set_title(plot_title, pad=12, fontweight='bold')
                        fig.tight_layout()
                        save_and_show(fig, plot_title, save_svg)

In [ ]:
def calculate_sample_average_with_se(dataframe, sample_ID_df):
    """
    Calculate sample averages and standard errors for each sample based on the given DataFrame and sample ID DataFrame.
    
    Parameters:
    dataframe (pd.DataFrame): The input DataFrame containing raw data.
    sample_ID_df (pd.DataFrame): The DataFrame containing sample IDs and related information.
    
    Returns:
    pd.DataFrame: A DataFrame containing sample averages and standard errors.
    """
    dataframe = dataframe.apply(pd.to_numeric, errors='coerce')
    dataframe.columns = dataframe.columns.astype(str)

    # Isolate columns containing each sample's keyword
    columns_dict = {}
    for sample in sample_ID_df.columns:
        columns_dict[sample] = [col for col in dataframe.columns if sample_ID_df.at['key', sample] in col]

    result_df = pd.DataFrame({'Elapsed': dataframe.loc[:, 'Elapsed']})

    # Calculate the average and standard error for selected columns at each timepoint
    for sample in sample_ID_df.columns:
        average_values = dataframe[columns_dict[sample]].mean(axis=1)
        avg_column_name = 'Average_' + sample_ID_df.at['key',sample]
        result_df[avg_column_name] = average_values
        se_values = dataframe[columns_dict[sample]].sem(axis=1)
        se_column_name = 'SE_' + sample_ID_df.at['key',sample]
        result_df[se_column_name] = se_values

    return result_df


In [ ]:
def fit_exp_growth_rolling(
    dataframe,
    sample_ID_df,
    ax=None,
    xlabel=None,
    ylabel=None,
    title=None,
    # windowing
    k_points=7,
    window_hours=None,
    min_points=5,
    search_bounds=(0.0, 100.0),
    # numerics
    eps=1e-6,
    annotate=True,
    # individual plot controls
    individual_mode=None,     # None | 'separate' | 'grid'
    grid_cols=3,              # columns per grid page
    grid_max_rows=5,          # max rows per grid page (pagination)
    manual_windows=None,
    force_start=None,
    max_window_hours=None
):
    """
    Estimates growth rates via log-linear fits chosen by max adj-R^2 (positive slope).
    - Fixed-count windows (k_points) by default for parity across samples.
    - Searches within `search_bounds`.
    - individual_mode:
        None       -> only combined plot
        'separate' -> one figure per sample (plt.show())
        'grid'     -> paginated grid; each page has <= grid_max_rows * grid_cols subplots
    """

    # --- helpers ---
    def get_color_for_column(col):
        for sample in sample_ID_df.columns:
            if str(sample_ID_df.at['key', sample]) in col:
                return sample_ID_df.at['color', sample]
        return None

    def fit_log_linear(xt, yt_log):
        if len(xt) < 2: return None
        m, c = np.polyfit(xt, yt_log, 1)
        if m <= 0: return None
        yhat = m*xt + c
        ss_res = np.sum((yt_log - yhat)**2)
        ss_tot = np.sum((yt_log - yt_log.mean())**2)
        r2 = 1.0 - ss_res/ss_tot if ss_tot > 0 else np.nan
        n = len(xt)
        adj_r2 = 1.0 - (1.0 - r2) * (n - 1) / max(n - 2, 1)
        return m, c, r2, adj_r2

    def resolve_start(col):
        if force_start is None:
            return None
        if isinstance(force_start, dict):
            return force_start.get(col, None)
        return float(force_start)

    # --- prep ---
    df = dataframe.apply(pd.to_numeric, errors='coerce').copy()
    if 'Elapsed' not in df.columns:
        raise ValueError("Need an 'Elapsed' column for time (hours).")

    out_index = ['a','b','t0','t1','r2','n_points']
    fitted_parameters_df = pd.DataFrame(index=out_index, columns=df.columns[2:], dtype=float)

    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 6))

    max_y = (df.iloc[:, 2:].max().max()) + 10
    lo, hi = search_bounds

    # collect sample columns first (used for pagination)
    sample_cols = list(df.columns[2:])

    # --- main loop over samples ---
    per_sample_results = {}  # cache results for plotting later in grid pages
    for col in sample_cols:
        t_full = df['Elapsed'].to_numpy()
        y_full = df[col].to_numpy()
        ok = np.isfinite(t_full) & np.isfinite(y_full) & (y_full > 0)
        t, y = t_full[ok], y_full[ok]
        order = np.argsort(t)
        t, y = t[order], y[order]
        ylog = np.log(y + eps)

        chosen = None  # (m, c, r2, adj, t0, t1, npts)

        # manual window override
        if manual_windows and col in manual_windows:
            t0, t1 = manual_windows[col]
            mask = (t >= t0) & (t <= t1)
            if mask.sum() >= 2:
                xt, yt = t[mask], ylog[mask]
                res = fit_log_linear(xt, yt)
                if res is not None:
                    m, c, r2, adj = res
                    chosen = (m, c, r2, adj, float(xt.min()), float(xt.max()), int(len(xt)))

        # forced start
        if chosen is None:
            t0_forced = resolve_start(col)
            if t0_forced is not None:
                start_idx = np.searchsorted(t, max(lo, t0_forced), side='left')
                if k_points is not None:
                    for end_idx in range(start_idx + k_points - 1, len(t)):
                        xt = t[start_idx:end_idx+1]
                        if len(xt) < k_points: continue
                        if xt[-1] > hi: break
                        yt = ylog[start_idx:end_idx+1]
                        res = fit_log_linear(xt, yt)
                        if res is None: continue
                        m, c, r2, adj = res
                        cand = (m, c, r2, adj, float(xt.min()), float(xt.max()), int(len(xt)))
                        if (chosen is None) or (adj > chosen[3]) or (np.isclose(adj, chosen[3]) and m > chosen[0]):
                            chosen = cand
                else:
                    minp = max(min_points, 2)
                    for end_idx in range(start_idx + minp - 1, len(t)):
                        xt = t[start_idx:end_idx+1]
                        if xt[-1] > hi: break
                        if max_window_hours is not None and (xt[-1] - xt[0]) > max_window_hours:
                            break
                        yt = ylog[start_idx:end_idx+1]
                        res = fit_log_linear(xt, yt)
                        if res is None: continue
                        m, c, r2, adj = res
                        cand = (m, c, r2, adj, float(xt.min()), float(xt.max()), int(len(xt)))
                        if (chosen is None) or (adj > chosen[3]) or (np.isclose(adj, chosen[3]) and m > chosen[0]):
                            chosen = cand

        # default auto: scan within [lo, hi]
        if chosen is None:
            if k_points is not None:
                for i in range(0, len(t) - k_points + 1):
                    xt = t[i:i+k_points]
                    if xt[0] < lo or xt[-1] > hi: continue
                    yt = ylog[i:i+k_points]
                    res = fit_log_linear(xt, yt)
                    if res is None: continue
                    m, c, r2, adj = res
                    cand = (m, c, r2, adj, float(xt.min()), float(xt.max()), int(len(xt)))
                    if (chosen is None) or (adj > chosen[3]) or (np.isclose(adj, chosen[3]) and m > chosen[0]):
                        chosen = cand
            else:
                if window_hours is None:
                    window_hours = 24.0
                j = 0
                for i in range(len(t)):
                    j = max(j, i)
                    while j < len(t) and (t[j] - t[i]) < window_hours:
                        j += 1
                    if j - i >= min_points:
                        xt = t[i:j]
                        if xt[0] < lo or xt[-1] > hi: continue
                        yt = ylog[i:j]
                        res = fit_log_linear(xt, yt)
                        if res is None: continue
                        m, c, r2, adj = res
                        cand = (m, c, r2, adj, float(xt.min()), float(xt.max()), int(len(xt)))
                        if (chosen is None) or (adj > chosen[3]) or (np.isclose(adj, chosen[3]) and m > chosen[0]):
                            chosen = cand

        if chosen is None:
            continue

        m, c, r2, adj, t0, t1, npts = chosen
        a_hat = float(np.exp(c))
        b_hat = float(m)

        fitted_parameters_df.at['a', col] = a_hat
        fitted_parameters_df.at['b', col] = b_hat
        fitted_parameters_df.at['t0', col] = t0
        fitted_parameters_df.at['t1', col] = t1
        fitted_parameters_df.at['r2', col] = r2
        fitted_parameters_df.at['n_points', col] = npts

        color = get_color_for_column(col) or None

        # combined plot
        ax.scatter(df['Elapsed'], df[col], label=f'Raw: {col}', color=color, s=18)
        x_fit = np.linspace(t0, t1, 100)
        y_fit = a_hat * np.exp(b_hat * x_fit)
        ax.plot(x_fit, y_fit, color=color, alpha=0.85, linewidth=2, label=f'Fit: {col}')
        if annotate:
            ax.text(x_fit[-1], y_fit[-1], f" b={b_hat:.3f}/h\nR^2={r2:.2f}",
                    ha='left', va='bottom', fontsize=8)

        # stash items for grid plotting later
        per_sample_results[col] = (color, x_fit, y_fit)

        # separate mode: show immediately
        if individual_mode == 'separate':
            fig_i, ax_i = plt.subplots(figsize=(7, 4))
            ax_i.scatter(df['Elapsed'], df[col], color=color, s=18, label=f'Raw: {col}')
            ax_i.plot(x_fit, y_fit, color=color, linewidth=2, alpha=0.95,
                      label=f'Fit {t0:.1f}-{t1:.1f} h (b={b_hat:.3f}/h, R^2={r2:.2f})')
            ax_i.set_xlabel(xlabel or 'Time (h)')
            ax_i.set_ylabel(ylabel or 'Signal')
            ax_i.set_title(col)
            ax_i.grid(True, alpha=0.25)
            ax_i.legend(fontsize=8)
            plt.show()

    # finalize combined plot
    ax.set_xlabel(xlabel or 'Time (h)')
    ax.set_ylabel(ylabel or 'Signal')
    ax.set_title(title or 'Exponential growth fit (fixed-count windows)')
    ax.set_ylim(0, max_y)
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

    # paginated grid mode
    if individual_mode == 'grid':
        n_per_page = max(int(grid_max_rows) * int(grid_cols), 1)
        cols = list(per_sample_results.keys())
        for start in range(0, len(cols), n_per_page):
            page_cols = cols[start:start + n_per_page]
            n_page = len(page_cols)
            rows = (n_page + grid_cols - 1) // grid_cols

            fig_g, axes_g = plt.subplots(rows, grid_cols, figsize=(4.8*grid_cols, 3.2*rows))
            # normalize axes to 2D array
            if rows == 1:
                axes_g = np.array([axes_g])
            if grid_cols == 1:
                axes_g = axes_g.reshape(rows, 1)

            # draw subplots
            for i, col in enumerate(page_cols):
                r, c = divmod(i, grid_cols)
                axg = axes_g[r, c]
                color, x_fit, y_fit = per_sample_results[col]
                axg.scatter(df['Elapsed'], df[col], color=color, s=14)
                axg.plot(x_fit, y_fit, color=color, linewidth=2, alpha=0.95)
                axg.set_title(col, fontsize=10)
                axg.set_xlabel(xlabel or 'Time (h)')
                axg.set_ylabel(ylabel or 'Signal')
                axg.grid(True, alpha=0.25)

            # turn off unused panels on last page
            total_slots = rows * grid_cols
            for i in range(n_page, total_slots):
                r, c = divmod(i, grid_cols)
                axes_g[r, c].axis('off')

            fig_g.tight_layout()
            plt.show()

    return fitted_parameters_df

In [ ]:
def isolate_sample_GRvalues(dataframe,sample_ID_df):
    dataframe = dataframe.apply(pd.to_numeric, errors='coerce')

    isolated_GR_df = pd.DataFrame(dtype=float)
    for sample in sample_ID_df.columns:
        isolated_GR_df[sample_ID_df.at['key',sample]] = None

    for column in dataframe.columns:
        for sample in sample_ID_df.columns:
            if sample_ID_df.at['key',sample] in column:
                pos = (isolated_GR_df[sample_ID_df.at['key',sample]].count())
                isolated_GR_df.loc[pos, sample_ID_df.at['key',sample]] = dataframe.loc['b', column]

    isolated_GR_df = isolated_GR_df.apply(pd.to_numeric, errors='coerce')

    print("Growth Rate Data for Samples")
    for column in isolated_GR_df.columns:
        average = isolated_GR_df[column].mean()
        standard_error = isolated_GR_df[column].sem()
        print(column)
        print("Average:", average)
        print("Standard Error:", standard_error)

    return isolated_GR_df

In [ ]:
def stats_and_plot(dataframe, alpha=0.05, xlabel=None, ylabel=None, title=None, sig_bars=True, point_label=False, fig_size=(6, 6), font_sz=18, save_svg=False):
    """
    Perform t-tests between data in each column of the DataFrame
    and create a box and whisker plot with indicators for statistical significance.

    Parameters:
    dataframe (pd.DataFrame): The input DataFrame.
    alpha (float, optional): The significance level for the t-tests. Default is 0.05.
    save_svg (bool): If True, saves the plot as an SVG file using the title as the filename.

    Returns:
    pd.DataFrame: A DataFrame containing the p-values between each pair of columns.
    """
    import matplotlib as mpl
    from scipy.stats import shapiro, levene, mannwhitneyu, ttest_ind, kruskal
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    from scipy.stats import f_oneway, tukey_hsd
    import scikit_posthocs as sp
    import pingouin as pg

    # --- Publication style settings ---
    # mpl.rcParams.update({
    #     'font.family': 'Arial',
    #     'font.weight': 'bold',
    #     'axes.labelweight': 'bold',
    #     'axes.titleweight': 'bold',
    #     'font.size': font_sz,
    #     'axes.titlesize': font_sz,
    #     'axes.labelsize': font_sz,
    #     'xtick.labelsize': font_sz-2,
    #     'ytick.labelsize': font_sz-2,
    #     'legend.fontsize': font_sz-4,
    #     'axes.linewidth': 1.5,
    #     'xtick.major.width': 1.5,
    #     'ytick.major.width': 1.5,
    #     'xtick.major.size': 5,
    #     'ytick.major.size': 5,
    #     'lines.linewidth': 1.5,
    # })

    # Convert dataframe to numeric
    dataframe = dataframe.apply(pd.to_numeric, errors='coerce')
    num_samples = dataframe.shape[1]
    p_values_df = pd.DataFrame(columns=dataframe.columns, index=dataframe.columns)

    fig, ax = plt.subplots(figsize=fig_size)

    # --- Clean spine styling ---
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.yaxis.set_ticks_position('left')
    ax.xaxis.set_ticks_position('bottom')

    # --- Boxplot ---
    bp = ax.boxplot(
        [dataframe[col].dropna() for col in dataframe.columns],
        labels=dataframe.columns,
        showmeans=True,
        showfliers=False,
        patch_artist=True,
        meanprops=dict(marker='D', markerfacecolor='black', markeredgecolor='black', markersize=5),
        medianprops=dict(color='black', linewidth=2),
        boxprops=dict(facecolor='white', color='black', linewidth=1.5),
        whiskerprops=dict(color='black', linewidth=1.5, linestyle='--'),
        capprops=dict(color='black', linewidth=1.5),
    )

    # --- Jittered data points ---
    for i, col in enumerate(dataframe.columns, start=1):
        y = dataframe[col].dropna()
        x = np.random.normal(i, 0.04, len(y))
        ax.scatter(x, y, color='dimgray', alpha=0.6, s=30, zorder=3, edgecolors='none')

        if point_label:
            valid_indices = dataframe[col].dropna().index
            for x_val, y_val, label in zip(x, y, valid_indices):
                ax.text(x_val, y_val, str(label), fontsize=8, ha='right', va='bottom')

    # --- Assumption testing ---
    alpha_assumption = 0.05
    groups = [dataframe[col].dropna().values for col in dataframe.columns]

    normality_ok = all(shapiro(g).pvalue > alpha_assumption for g in groups if len(g) >= 3)
    levene_p = levene(*groups).pvalue
    variance_ok = levene_p > alpha_assumption

    print(f"Normality (Shapiro-Wilk):     {'PASS' if normality_ok else 'FAIL'}")
    print(f"Equal variance (Levene's):    {'PASS' if variance_ok else 'FAIL'}")

    # --- Statistics ---
    if num_samples == 2:
        col1, col2 = dataframe.columns[0], dataframe.columns[1]
        g1, g2 = dataframe[col1].dropna().values, dataframe[col2].dropna().values

        if normality_ok and variance_ok:
            test_used = "Independent samples t-test"
            _, p_value = ttest_ind(g1, g2, equal_var=True)
        elif normality_ok and not variance_ok:
            test_used = "Welch's t-test"
            _, p_value = ttest_ind(g1, g2, equal_var=False)
        else:
            test_used = "Mann-Whitney U test"
            _, p_value = mannwhitneyu(g1, g2, alternative='two-sided')

        p_values_df.loc[col1, col2] = p_value
        print(f"Test selected:                {test_used}")
        print(f"p-value:                      {p_value:.4f}")

    else:
        if normality_ok and variance_ok:
            test_used = "One-way ANOVA + Tukey HSD"
            _, p_omnibus = f_oneway(*groups)
            res = tukey_hsd(*groups)
            p_values_df = pd.DataFrame(res.pvalue, columns=dataframe.columns, index=dataframe.columns)

        elif normality_ok and not variance_ok:
            test_used = "Welch's ANOVA + Games-Howell"
            melted = dataframe.melt(var_name='group', value_name='value').dropna()
            welch_result = pg.welch_anova(data=melted, dv='value', between='group')
            p_omnibus = welch_result['p_unc'].values[0]
            gh = pg.pairwise_gameshowell(data=melted, dv='value', between='group')
            p_values_df = pd.DataFrame(np.nan, columns=dataframe.columns, index=dataframe.columns)
            for _, row in gh.iterrows():
                p_values_df.loc[row['A'], row['B']] = row['pval']
                p_values_df.loc[row['B'], row['A']] = row['pval']

        else:
            test_used = "Kruskal-Wallis + Dunn's post-hoc (Bonferroni)"
            _, p_omnibus = kruskal(*groups)
            melted = dataframe.melt(var_name='group', value_name='value').dropna()
            dunn = sp.posthoc_dunn(melted, val_col='value', group_col='group', p_adjust='bonferroni')
            p_values_df = dunn

        print(f"Test selected:                {test_used}")
        print(f"Omnibus p-value:              {p_omnibus:.2e}")

    # --- Significance bars ---
    if sig_bars:
        y_max = dataframe.max().max()
        y_min_ax, y_max_ax = ax.get_ylim()
        y_span = y_max_ax - y_min_ax
        line_offset = y_span * 0.03
        text_offset = y_span * 0.01
        drawn_pairs = set()
        highest_y = y_max

        for col1 in p_values_df.columns:
            for col2 in p_values_df.index:
                p_value = p_values_df.loc[col2, col1]
                if not pd.isna(p_value) and p_value < alpha:
                    if (col1, col2) in drawn_pairs or (col2, col1) in drawn_pairs:
                        continue

                    if p_value < 0.001:
                        sig_symbol = '***'
                    elif p_value < 0.01:
                        sig_symbol = '**'
                    elif p_value < 0.05:
                        sig_symbol = '*'
                    else:
                        continue

                    x1 = dataframe.columns.get_loc(col1) + 1
                    x2 = dataframe.columns.get_loc(col2) + 1
                    y = y_max + line_offset

                    ax.plot([x1, x2], [y, y], lw=1.5, color='black')
                    ax.text((x1 + x2) * 0.5, y + text_offset, sig_symbol,
                            ha='center', va='bottom', color='black', fontsize=13)

                    drawn_pairs.add((col1, col2))
                    drawn_pairs.add((col2, col1))
                    line_offset += y_span * 0.09
                    highest_y = y + text_offset

        # Expand y-axis to fit all bars with a small buffer
        ax.set_ylim(bottom=dataframe.min().min() * 0.95, top=highest_y * 1.02)
        
    # --- Labels & title ---
    ax.set_xlabel(xlabel if xlabel else 'Samples', labelpad=10)
    ax.set_ylabel(ylabel if ylabel else 'Growth Rate', labelpad=10)
    plot_title = title if title else 'Boxplot with Statistical Significance Indicators'
    ax.set_title(plot_title, pad=12, fontweight='bold')

    fig.tight_layout()

    if save_svg:
        filename = f"{plot_title}.svg"
        fig.savefig(filename, format='svg', bbox_inches='tight', dpi=300)

    plt.show()
    return p_values_df

In [ ]:
def split_samples_by_type(df):
    # Calculate the average of each column and convert to a dataframe with a single row
    sample_GR_values_avg = df.mean().to_frame().T

    df_F = pd.DataFrame(columns=['Fusion'])
    df_C = pd.DataFrame(columns=['Control'])
    df_P = pd.DataFrame(columns=['Parental'])

    for col in sample_GR_values_avg.columns:
        if 'G' in col or 'MC' in col:
            df_P = pd.concat([df_P, sample_GR_values_avg[[col]].rename(columns={col: 'Parental'}).set_index(pd.Index([col]))], axis=0)
        elif 'F' in col:
            df_F = pd.concat([df_F, sample_GR_values_avg[[col]].rename(columns={col: 'Fusion'}).set_index(pd.Index([col]))], axis=0)
        elif 'C' in col:
            df_C = pd.concat([df_C, sample_GR_values_avg[[col]].rename(columns={col: 'Control'}).set_index(pd.Index([col]))], axis=0)
        else:
            print(f"Column '{col}' does not match any condition.")

    # Combine the filtered dataframes into a new dataframe with simple column names
    fusion_ctrl_parent_GR = pd.concat([df_F, df_C, df_P], axis=1)

    if df_F.empty:
        print("Warning: df_F (Fusion) is empty.")
    if df_C.empty:
        print("Warning: df_C (Control) is empty.")
    if df_P.empty:
        print("Warning: df_P (Parental) is empty.")

    return sample_GR_values_avg, fusion_ctrl_parent_GR

## Main Function

In [ ]:
# Config: (variable name, incucyte file path)
incucyte_base = '/stor/work/Brock/kennedy/SC_repo/data/GrowthRate/HCC1806_GrowthRateData/all_1806_conf_data'

sample_files = [
    ('f5_incucyte_df', f'{incucyte_base}/KH2404_F5_PHASE_V4.txt'),
    ('f2_incucyte_df', f'{incucyte_base}/KH2404_F2_ph_V4.txt'),
    ('f3_incucyte_df', f'{incucyte_base}/KH2404_F3_ph_V4.txt'),
    ('f1_incucyte_df', f'{incucyte_base}/KH2404_F1_ph_V4.txt'),
    ('f4_incucyte_df', f'{incucyte_base}/KH2404_F4_PHASE_V4.txt'),
    ('f6_incucyte_df', f'{incucyte_base}/KH2404_F6_ph_V4.txt'),
    ('f7_incucyte_df', f'{incucyte_base}/KH2404_F7_PHASE_V4.txt'),
    ('f8_incucyte_df', f'{incucyte_base}/KH2404_F8_ph_V4.txt'),
    ('c1_incucyte_df', f'{incucyte_base}/KH2404_C1_ph_V4.txt'),
    ('c2_incucyte_df', f'{incucyte_base}/KH2404_C2_ph_V4.txt'),
    ('c3_incucyte_df', f'{incucyte_base}/KH2404_C3_ph_V4.txt'),
    ('c4_incucyte_df', f'{incucyte_base}/KH2404_C4_PHASE_V4.txt'),
    ('c5_incucyte_df', f'{incucyte_base}/KH2404_C5_ph_V4.txt'),
    ('c6_incucyte_df', f'{incucyte_base}/KH2404_C6_ph_V4.txt'),
    ('c7_incucyte_df', f'{incucyte_base}/KH2404_C7_ph_V4.txt'),
    ('c8_incucyte_df', f'{incucyte_base}/KH2404_C8_PHASE_V4.txt'),
    ('MC_P_incucyte_df', f'{incucyte_base}/KH2404_MC_ph_V4.txt'),
    ('GFP_P_incucyte_df', f'{incucyte_base}/KH2404_GFP_ph_V4.txt'),
]

# Import every sample. The first one in the list keeps 'Elapsed'/'Date Time'
# so raw_incucyte_df ends up with those columns exactly once.
sample_dfs = {}
for i, (var_name, og_file) in enumerate(sample_files):
    sample_dfs[var_name] = import_incucyte_sample(og_file, keep_all_columns=(i == 0))
    globals()[var_name] = sample_dfs[var_name]

raw_incucyte_df = pd.concat(list(sample_dfs.values()), axis=1)

import sample ID file for color selection and sample naming

In [ ]:
sample_ID_df = pd.read_csv('/stor/work/Brock/kennedy/SC_repo/data/GrowthRate/HCC1806_GrowthRateData/sample_ID_1806.csv',index_col=0)

Next, use the plot_variable_vs_time(dataframe, sample_ID_df, xlabel=None, ylabel=None, title=None) function to plot each individual well of incucyte data vs. time. IMPORTANT: Change the values for xlabel, ylabel, and title to your desired inputs.

In [ ]:
plot_variable_vs_time(raw_incucyte_df, sample_ID_df, xlabel='Time (hours)', ylabel='Fluorescent Count', title='Raw Data - Count vs. Time')

Next, use the calculate_sample_average_with_se(dataframe,sample_ID_df) function to create a dataframe with the average of each sample group values at each timepoint. 

In [ ]:
sample_averages_df = calculate_sample_average_with_se(raw_incucyte_df,sample_ID_df)

Now we can once again use the plot_variable_vs_time(dataframe, sample_ID_df, xlabel=None, ylabel=None, title=None) function to plot the averaged data of the exponential regions.

In [ ]:
plot_variable_vs_time(sample_averages_df.iloc[2:35,:-4],sample_ID_df, grid_lines=False, save_svg=True, xlabel='Time (hours)', ylabel='Confluency (%)', title='Sample Confluence vs. Time', plot_separately=False)

In [ ]:
exp_coeff_values= fit_exp_growth_rolling(
    raw_incucyte_df,
    sample_ID_df,
    ax=None,
    xlabel=None,
    ylabel=None,
    title=None,
    # windowing
    k_points=13,
    window_hours=None,
    min_points=5,
    search_bounds=(12.0, 64.0),
    # numerics
    eps=1e-6,
    annotate=True,
    # individual plot controls
    individual_mode=None,     # None | 'separate' | 'grid'
    grid_cols=3,              # columns per grid page
    grid_max_rows=5,          # max rows per grid page (pagination)
    manual_windows=None,
    force_start=None,
    max_window_hours=None
)

Finally, the isolate_sample_GRvalues(dataframe,sample_ID_df) function will be used to isolate the GR values for each respective sample into their own column in a new dataframe. Then, the stats_and_plot(dataframe, alpha=0.05, xlabel=None, ylabel=None, title=None, sig_bars=True) function will be used on the dataframe with the isolated values to run t-tests between samples, output those as a new dataframe, and to create a box and whisker plot showing statistical significance (if any) between samples.

In [ ]:
sample_GR_values = isolate_sample_GRvalues(exp_coeff_values,sample_ID_df)
sample_GR_values_avg, fusion_ctrl_parent_GR = split_samples_by_type(sample_GR_values)
f_c_p_p_values = stats_and_plot(fusion_ctrl_parent_GR, alpha=0.05, sig_bars=True, point_label=True, fig_size=(6, 6),title='Growthrate Comparison of HCC1806 Control vs. Fusion vs. Parental')
f_c_p_values = stats_and_plot(fusion_ctrl_parent_GR[['Fusion','Control']], alpha=0.05, sig_bars=True, point_label=False, save_svg=True, fig_size=(6, 6),title='Growthrate Comparison of HCC1806 Fusion vs. Control Clones')

print("Growth Rate Data for Samples by Group")
for column in fusion_ctrl_parent_GR.columns:
    average = fusion_ctrl_parent_GR[column].mean()
    standard_error = fusion_ctrl_parent_GR[column].sem()
    standard_dev = fusion_ctrl_parent_GR[column].std()
    print(column)
    print("Average:", average)
    print("Standard Error:", standard_error)
    print("Standard Deviation:", standard_dev,'\n')

print (f_c_p_values)

In [ ]:
DT_1806_values = (np.log(2) / fusion_ctrl_parent_GR.astype(float))
DT_f_c_p_p_values = stats_and_plot(DT_1806_values, alpha=0.05, sig_bars=True, point_label=True, ylabel='Doubling Time (h)', fig_size=(6, 6),title='Doubling Time Comparison of HCC1806 Control vs. Fusion vs. Parental')
DT_f_c_p_values = stats_and_plot(DT_1806_values[['Control','Fusion']], alpha=0.05, font_sz=24, sig_bars=True, save_svg=True, ylabel='Doubling Time (h)', point_label=False, fig_size=(6, 6),title='Doubling Time Comparison of HCC1806 Fusion vs. Control Clones')

print("Growth Rate Data for Samples by Group")
for column in DT_1806_values.columns:
    average = DT_1806_values[column].mean()
    standard_error = DT_1806_values[column].sem()
    standard_dev = DT_1806_values[column].std()
    print(column)
    print("Average:", average)
    print("Standard Error:", standard_error)
    print("Standard Deviation:", standard_dev,'\n')

print(DT_f_c_p_values)